# Frecuencia real de refresco de las cámaras

**Motivación.** Un ciclo semafórico urbano dura ~60–120 s. Si capturamos una imagen cada 30–60 s solo obtenemos 1–2 muestras por ciclo: no se pueden medir duraciones de fase (OE-4) ni seguir peatones entre fotogramas. Antes de rediseñar la captura hay que saber **cada cuánto publica el servidor una imagen nueva**.

**Método.** El servidor de imágenes (`ctraficomovilidad.malaga.eu`) devuelve la cabecera HTTP `Last-Modified` en las peticiones GET (no en HEAD). Se consultan varias cámaras cada `POLL_S` segundos y se registra:

- `Last-Modified`: instante exacto en que el servidor publicó la imagen.
- SHA-1 del contenido: método de respaldo si la cabecera faltara (precisión limitada a `POLL_S`).

El código de medición está en `scripts/medir_refresco.py` (también se puede lanzar desde terminal). Además, `captura_camaras.py` guarda ahora `server_last_modified` en cada captura, por lo que el refresco también se puede analizar de forma pasiva sobre los metadatos de la captura continua (sección 4).

**Decisión que se toma con este notebook:**

| Refresco mediano | Consecuencia para OE-4 |
|---|---|
| ≤ 5 s | Captura de alta frecuencia en 3–5 cámaras de referencia → medir duraciones de fase directamente. |
| 5–20 s | Duraciones con incertidumbre ± refresco; combinar con el enfoque estadístico. |
| > 20 s | Enfoque estadístico: fracción de rojo/verde a partir de muchas muestras con fase aleatoria. |

## 1. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import medir_refresco as mr
from config import load_config, path

CAMERAS = load_config()["refresco"]["camaras_referencia"]  # config.yaml
DURATION_MIN = 10   # minutos de medición
POLL_S = 1.0        # segundos entre consultas
SAVE_IMAGES = True  # guarda cada imagen nueva (útil para ver el ciclo semafórico a alta frecuencia)

print("Salida:", mr.OUT_DIR)

## 2. Medición activa

Tarda `DURATION_MIN` minutos. Si ya existe una medición previa, se puede saltar esta celda y cargar el CSV en la siguiente.

In [ ]:
polls, summary = mr.measure(CAMERAS, DURATION_MIN, POLL_S, save_images=SAVE_IMAGES)
display(summary)

In [ ]:
# Cargar la última medición disponible (permite reanalizar sin volver a medir)
polls_file = sorted(mr.OUT_DIR.glob("refresco_*Z.csv"))[-1]
polls = pd.read_csv(polls_file, parse_dates=["requested_at", "last_modified"])
print("Medición:", polls_file.name, "| consultas:", len(polls))

errores = polls[polls["status"].astype(str) != "200"]
print(f"Consultas con error: {len(errores)} ({100 * len(errores) / len(polls):.1f}%)")

intervals = mr.change_intervals(polls)
summary = mr.summarize(polls, POLL_S)
display(summary)

## 3. Distribución de intervalos entre imágenes nuevas

Si ambos métodos coinciden, `Last-Modified` es fiable y se puede usar como marca temporal real de la imagen (más precisa que la hora de descarga).

In [ ]:
if intervals.empty:
    print("No se detectaron cambios de imagen: amplía DURATION_MIN.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.boxplot(data=intervals, x="camera_id", y="interval_s", hue="method", ax=axes[0])
    axes[0].set_title("Intervalo entre imágenes nuevas por cámara")
    axes[0].set_ylabel("segundos")

    sns.histplot(data=intervals[intervals["method"] == "last_modified"], x="interval_s", bins=30, ax=axes[1])
    axes[1].set_title("Histograma (Last-Modified, todas las cámaras)")
    axes[1].set_xlabel("segundos")
    plt.tight_layout()
    plt.show()

    refresco = intervals[intervals["method"] == "last_modified"]["interval_s"].median()
    print(f"Refresco mediano global: {refresco:.1f} s")

## 4. Análisis pasivo sobre la captura continua

Con los metadatos de `captura_camaras.py` (columna `server_last_modified`) se obtiene la **antigüedad** de cada imagen en el momento de descargarla y si dos capturas consecutivas son la misma imagen (captura redundante).

In [ ]:
meta_files = sorted(path("metadata").glob("capturas_*.csv"))
meta = pd.concat([pd.read_csv(f) for f in meta_files], ignore_index=True) if meta_files else pd.DataFrame()

if meta.empty or meta.get("server_last_modified", pd.Series(dtype=str)).isna().all():
    print("Aún no hay capturas con server_last_modified (las capturas anteriores a la migración UTC no lo registraban).")
else:
    meta = meta[meta["status"] == "OK"].copy()
    meta["capture_time"] = pd.to_datetime(meta["capture_time"], utc=True)
    meta["server_last_modified"] = pd.to_datetime(meta["server_last_modified"], utc=True)
    meta = meta.dropna(subset=["server_last_modified"]).sort_values(["camera_id", "capture_time"])

    meta["antiguedad_s"] = (meta["capture_time"] - meta["server_last_modified"]).dt.total_seconds()
    meta["repetida"] = meta.groupby("camera_id")["server_last_modified"].diff().dt.total_seconds().eq(0)

    print(f"Capturas analizadas: {len(meta)}")
    print(f"Antigüedad mediana de la imagen al descargarla: {meta['antiguedad_s'].median():.1f} s")
    print(f"Capturas repetidas (misma imagen que la anterior): {100 * meta['repetida'].mean():.1f}%")
    display(meta.groupby("camera_id")["antiguedad_s"].describe())

## 5. Conclusión

> Completar tras la medición: refresco mediano observado, variabilidad entre cámaras y decisión adoptada para OE-4 según la tabla de la introducción (frecuencia de captura en `config.yaml` → `captura`).

El resumen guardado (`refresco_*_resumen.csv`) lo usan después `02_preprocess_labelled_basic` (intervalo de muestreo de fotogramas) y `05_evaluation` (decide si se pueden medir duraciones de fase).